# Lab 12 — Plan-and-execute from scratch

Build a planner-executor system. A planner agent emits a structured
`Plan`; a supervisor resolves dependencies and dispatches steps to a
bounded executor pool; bounded replanning on failure.

No frameworks. The dispatcher is plain Python — once the plan exists,
dependency resolution is mechanical. Reuses Lab 10's web tools at the
executor level.

> ⏱ Run time: 120-150 min including reading.
> 📖 Read [`concepts/multi-agent/plan-and-execute.md`](../../concepts/multi-agent/plan-and-execute.md)
> and [`concepts/multi-agent/planner-executor-pattern.md`](../../concepts/multi-agent/planner-executor-pattern.md)
> first. The lab references the four failure modes and the five
> planner-prompt rules directly.

## Step 0: Setup

Same setup as Labs 10/11. No new dependencies beyond stdlib —
`concurrent.futures.ThreadPoolExecutor` is built into Python.

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


**Sample output:**

```
Using openai / gpt-4o-mini
```

## Step 1: Recap of Lab 10/11 machinery

Lab 12 reuses:

- `chat_with_tools` — provider-agnostic LLM client (unchanged).
- `web_search`, `fetch_page` — Lab 10's web tools (unchanged).
- `_action_hash` — Lab 03/10/11's dedup helper (unchanged).
- `StrictModel` — Lab 02's `extra="forbid"` Pydantic base.

We re-state them inline here. **Lab 12 adds new components on top of
these primitives, not new versions of them.**

### The chat client and supporting primitives

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
    temperature: float = 0,
) -> AssistantMessage:
    """Provider-agnostic chat client. Same as Lab 10/11."""
    if PROVIDER == "openai":
        from openai import OpenAI

        resp = OpenAI().chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(
                    id=tc.id,
                    name=tc.function.name,
                    arguments=json.loads(tc.function.arguments),
                )
                for tc in (msg.tool_calls or [])
            ],
        )

    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {
                "name": t["function"]["name"],
                "description": t["function"]["description"],
                "input_schema": t["function"]["parameters"],
            }
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            tools=anth_tools or None,
            max_tokens=2048,
            temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [
            ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
            for b in resp.content
            if getattr(b, "type", None) == "tool_use"
        ]
        return AssistantMessage(content=text or None, tool_calls=tcs)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    """Lab 03/10/11 action-hash dedup helper."""
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    """Lab 02 pattern: extra='forbid' so schemas reject unexpected fields."""
    model_config = ConfigDict(extra="forbid")


def _strip_code_fences(raw: str) -> str:
    """Defensive JSON extraction. Models sometimes wrap JSON in ```json fences."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    return raw


### Web tools (Lab 10, unchanged)

We import these here for completeness. Identical to Lab 10/11; skip
reading if you've done either of those.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

RecencyType = Literal["any", "day", "week", "month", "year"]
_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = (
    "AgenticAIEngineer-CourseLab/0.1 "
    "(https://github.com/MHHamdan/Agentic-AI-Engineer)"
)
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit",
    "register to read",
]


def web_search(query: str, recency: RecencyType = "any", max_results: int = 8) -> dict:
    """Same as Lab 03/10/11."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other", "detail": f"{type(e).__name__}: {e}"}
    if not raw:
        return {"status": "empty", "query": query, "detail": "no results returned"}
    return {
        "status": "ok",
        "results": [
            {"title": (r.get("title") or "").strip(),
             "url": (r.get("href") or "").strip(),
             "snippet": (r.get("body") or "").strip()}
            for r in raw if r.get("href")
        ][:max_results],
    }


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    """Same as Lab 03/10/11."""
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout",
                "detail": "request timed out after 15s"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()
    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected in body"}
    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "total_chars": len(text)}
    return {"status": "ok", "url": url, "title": title, "text": text}


## Step 2: The `Plan` schema

The first new component. A `Plan` is a list of `PlanStep`s. Each step
has an `id`, a `description`, the `tool` to use, its `args`, a list of
step IDs in `depends_on`, and an optional `parallel_group`.

`StrictModel(extra="forbid")` means the planner cannot emit fields
outside this schema — anything extra fails Pydantic validation, which
gets surfaced back to the planner as a structured retry signal.

In [ ]:
class PlanStep(StrictModel):
    """One step in a Plan. Atomic: one tool call per step."""
    id: str = Field(description="Unique step ID, e.g. 'step_1'")
    description: str = Field(
        description="What this step does, in self-contained natural language."
    )
    tool: str = Field(
        description="Which tool to invoke. Must be in the executor's tool registry."
    )
    args: dict = Field(
        default_factory=dict,
        description="Arguments to pass to the tool.",
    )
    depends_on: list[str] = Field(
        default_factory=list,
        description="IDs of steps whose output this step needs.",
    )
    parallel_group: str | None = Field(
        default=None,
        description=(
            "Optional grouping for concurrent execution. Steps with the "
            "same parallel_group can run concurrently (dependencies "
            "permitting). None means run alone."
        ),
    )


class Plan(StrictModel):
    """A list of PlanSteps. Validates dependency-graph integrity."""
    steps: list[PlanStep] = Field(description="The ordered list of steps.")

    def validate_graph(self, available_tools: set[str]) -> list[str]:
        """Return a list of validation errors (empty list = valid plan)."""
        errors: list[str] = []
        step_ids = {s.id for s in self.steps}
        if len(step_ids) != len(self.steps):
            errors.append("duplicate step IDs")
        for s in self.steps:
            if s.tool not in available_tools:
                errors.append(
                    f"step {s.id} uses tool '{s.tool}' not in executor tool registry: "
                    f"{sorted(available_tools)}"
                )
            for dep in s.depends_on:
                if dep not in step_ids:
                    errors.append(f"step {s.id} depends on unknown step '{dep}'")
                if dep == s.id:
                    errors.append(f"step {s.id} depends on itself")
        # Parallel groups must not contain transitive dependencies
        by_group: dict[str, list[PlanStep]] = {}
        for s in self.steps:
            if s.parallel_group is not None:
                by_group.setdefault(s.parallel_group, []).append(s)
        for group_name, group_steps in by_group.items():
            group_ids = {s.id for s in group_steps}
            for s in group_steps:
                # A step in a parallel group cannot depend on another step in
                # the same group (the dispatcher would deadlock or serialize).
                for dep in s.depends_on:
                    if dep in group_ids:
                        errors.append(
                            f"parallel_group '{group_name}' contains step {s.id} "
                            f"which depends on group-mate {dep}"
                        )
        # Cycle detection (Kahn's algorithm)
        incoming = {s.id: set(s.depends_on) for s in self.steps}
        no_incoming = [sid for sid, deps in incoming.items() if not deps]
        visited: list[str] = []
        while no_incoming:
            n = no_incoming.pop()
            visited.append(n)
            for sid, deps in incoming.items():
                if n in deps:
                    deps.discard(n)
                    if not deps and sid not in visited and sid not in no_incoming:
                        no_incoming.append(sid)
        if len(visited) < len(self.steps):
            unvisited = [s.id for s in self.steps if s.id not in visited]
            errors.append(f"cycle detected involving steps: {unvisited}")
        return errors


# Limits
MAX_PLAN_STEPS = 8
MAX_PARALLEL_EXECUTORS = 3
EXECUTOR_MAX_STEPS = 4
MAX_REPLANS = 2
SUPERVISOR_MAX_STEPS = 12

print(f"PlanStep schema fields: {list(PlanStep.model_fields.keys())}")
print(f"Plan schema fields:     {list(Plan.model_fields.keys())}")
print(f"Caps: MAX_PLAN_STEPS={MAX_PLAN_STEPS}, "
      f"MAX_PARALLEL_EXECUTORS={MAX_PARALLEL_EXECUTORS}, "
      f"EXECUTOR_MAX_STEPS={EXECUTOR_MAX_STEPS}, "
      f"MAX_REPLANS={MAX_REPLANS}, "
      f"SUPERVISOR_MAX_STEPS={SUPERVISOR_MAX_STEPS}")


**Sample output:**

```
PlanStep schema fields: ['id', 'description', 'tool', 'args', 'depends_on', 'parallel_group']
Plan schema fields:     ['steps']
Caps: MAX_PLAN_STEPS=8, MAX_PARALLEL_EXECUTORS=3, EXECUTOR_MAX_STEPS=4, MAX_REPLANS=2, SUPERVISOR_MAX_STEPS=12
```

### Sanity-check the graph validator

A quick test before trusting it with real planner output:

In [ ]:
# Valid plan: 3 steps, step_2 and step_3 in parallel group "A", both depend on step_1
valid_plan = Plan(steps=[
    PlanStep(id="step_1", description="Search the web for X.",
             tool="web_search", args={"query": "X"}, depends_on=[]),
    PlanStep(id="step_2", description="Fetch the first URL from step_1.",
             tool="fetch_page", args={"url": "..."},
             depends_on=["step_1"], parallel_group="fetches"),
    PlanStep(id="step_3", description="Fetch the second URL from step_1.",
             tool="fetch_page", args={"url": "..."},
             depends_on=["step_1"], parallel_group="fetches"),
])

errors = valid_plan.validate_graph({"web_search", "fetch_page"})
print(f"Valid plan: {len(errors)} errors  (expect 0)")

# Invalid: cycle
cyclic_plan = Plan(steps=[
    PlanStep(id="step_1", description="x", tool="web_search",
             depends_on=["step_2"]),
    PlanStep(id="step_2", description="y", tool="web_search",
             depends_on=["step_1"]),
])
errors = cyclic_plan.validate_graph({"web_search"})
print(f"Cyclic plan errors: {errors}")

# Invalid: parallel group with internal dependency
deadlock_plan = Plan(steps=[
    PlanStep(id="step_1", description="x", tool="web_search",
             parallel_group="A"),
    PlanStep(id="step_2", description="y", tool="web_search",
             depends_on=["step_1"], parallel_group="A"),
])
errors = deadlock_plan.validate_graph({"web_search"})
print(f"Bad parallel group errors: {errors}")

# Invalid: unknown tool
bad_tool_plan = Plan(steps=[
    PlanStep(id="step_1", description="x", tool="nonexistent_tool"),
])
errors = bad_tool_plan.validate_graph({"web_search", "fetch_page"})
print(f"Bad tool errors: {errors}")


**Sample output:**

```
Valid plan: 0 errors  (expect 0)
Cyclic plan errors: ['cycle detected involving steps: [\'step_1\', \'step_2\']']
Bad parallel group errors: ["parallel_group 'A' contains step step_2 which depends on group-mate step_1"]
Bad tool errors: ["step step_1 uses tool 'nonexistent_tool' not in executor tool registry: ['fetch_page', 'web_search']"]
```

The validator catches all four classes of plan malformation: cycles,
parallel-group violations, unknown tools, and (not shown) duplicate IDs
and unknown dependency references.

## Step 3: The planner agent

The planner emits a `Plan` as JSON. The system prompt encodes the five
planner-prompt rules from the
[planner-executor-pattern concept page](../../concepts/multi-agent/planner-executor-pattern.md#the-five-planner-prompt-rules):

1. Steps must be atomic (one tool call per step).
2. Dependencies must be explicit.
3. Parallel groups must be honestly independent.
4. Step descriptions must be self-contained.
5. Plans must be bounded (`MAX_PLAN_STEPS = 8`).

The executor's tool registry is passed into the system prompt — this
closes the **plan-execution gap** failure mode. The planner literally
cannot emit a step using a tool the executor doesn't have, because
validation will reject the plan and the retry loop will surface the
error.

In [ ]:
# Executor's tool registry. Lab 12 exposes Lab 10's web tools at the executor level.
EXECUTOR_TOOL_REGISTRY: dict[str, dict] = {
    "web_search": {
        "description": (
            "Search the web. Returns up to max_results items with title, url, "
            "snippet. Use this to find URLs worth fetching."
        ),
        "args_schema": {
            "query": "string, 3-8 specific words",
            "recency": "one of: any, day, week, month, year (default: any)",
            "max_results": "integer 1-10 (default: 8)",
        },
    },
    "fetch_page": {
        "description": (
            "Fetch the full text content of a single URL. Use after web_search "
            "to read the most relevant pages."
        ),
        "args_schema": {
            "url": "string, the URL to fetch",
            "max_chars": "integer (default: 8000)",
        },
    },
}


def _format_tool_registry(registry: dict[str, dict]) -> str:
    """Render the registry for the planner's system prompt."""
    lines = []
    for tool_name, info in registry.items():
        lines.append(f"- {tool_name}: {info['description']}")
        for arg_name, arg_desc in info["args_schema"].items():
            lines.append(f"    args.{arg_name}: {arg_desc}")
    return "\n".join(lines)


PLANNER_SYSTEM_PROMPT_TEMPLATE = """You are a planner agent. Given a user task,
emit a Plan as a JSON object with this schema:

{{
  "steps": [
    {{
      "id": "step_1",
      "description": "What this step does, self-contained.",
      "tool": "tool_name",
      "args": {{...}},
      "depends_on": ["step_id_1", ...],
      "parallel_group": "group_name" or null
    }},
    ...
  ]
}}

The EXECUTOR has access to these tools (and only these tools):
{tool_registry}

RULES (each prevents a specific failure mode):

1. ATOMIC STEPS. One tool call per step. If a step says "search and read,"
   split it into "search" and "read."

2. EXPLICIT DEPENDENCIES. Every step that uses another step's output must
   list that step ID in depends_on.

3. HONEST PARALLEL GROUPS. Steps in the same parallel_group must NOT depend
   on each other (directly or transitively). If two steps could be reordered
   without changing the outcome, they can share a parallel_group.

4. SELF-CONTAINED DESCRIPTIONS. The executor only sees one step at a time
   plus its dependencies' outputs. Don't reference "the analysis from step 3"
   without explaining what that means.

5. BOUNDED PLANS. At most {max_steps} steps. If a task needs more, the last
   step should produce a checklist for the next plan.

DO NOT:
- Use any tool not in the registry above.
- Create circular dependencies.
- Put more than {max_steps} steps in one plan.
- Make up tool argument names; use only the args.* names listed.

Return ONLY valid JSON. No prose preamble. No markdown fences. Just JSON.
"""


def _build_planner_system_prompt() -> str:
    return PLANNER_SYSTEM_PROMPT_TEMPLATE.format(
        tool_registry=_format_tool_registry(EXECUTOR_TOOL_REGISTRY),
        max_steps=MAX_PLAN_STEPS,
    )


def planner_agent(
    task: str,
    failure_context: dict | None = None,
    max_retries: int = 2,
) -> Plan | dict:
    """Emit a Plan for the given task.

    Returns a validated Plan on success, or a structured error envelope
    if validation fails after max_retries attempts.

    If failure_context is provided (from a previous failed execution),
    it's included in the user prompt as a "previous attempt failed
    because X" signal.
    """
    system_prompt = _build_planner_system_prompt()
    user_prompt = f"USER TASK:\n{task}"
    if failure_context:
        user_prompt += (
            f"\n\nA PREVIOUS PLAN FAILED. Revise:\n"
            f"failed_step_id: {failure_context.get('step_id')}\n"
            f"error: {failure_context.get('error')}\n"
            f"completed_steps: {failure_context.get('completed_steps', [])}\n"
            f"Produce a new plan that avoids the failure. You may reuse "
            f"completed step outputs implicitly (the executor still has them)."
        )

    last_error: str = "no attempt made"
    for _attempt in range(max_retries + 1):
        msg = chat_with_tools(
            [{"role": "system", "content": system_prompt},
             {"role": "user", "content": user_prompt}],
            temperature=0,
        )
        raw = _strip_code_fences(msg.content or "")
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError as e:
            last_error = f"JSON parse error: {e}"
            user_prompt = (
                f"{user_prompt}\n\nYour previous response was not valid JSON: "
                f"{last_error}. Return ONLY a JSON object."
            )
            continue

        try:
            plan = Plan.model_validate(obj)
        except ValidationError as e:
            last_error = f"schema validation error: {e}"
            user_prompt = (
                f"{user_prompt}\n\nYour previous plan failed Pydantic "
                f"validation: {last_error}. Fix the schema and try again."
            )
            continue

        graph_errors = plan.validate_graph(set(EXECUTOR_TOOL_REGISTRY.keys()))
        if graph_errors:
            last_error = "; ".join(graph_errors)
            user_prompt = (
                f"{user_prompt}\n\nYour previous plan had dependency-graph "
                f"errors: {last_error}. Fix the graph and try again."
            )
            continue

        if len(plan.steps) > MAX_PLAN_STEPS:
            last_error = f"plan has {len(plan.steps)} steps; max is {MAX_PLAN_STEPS}"
            user_prompt = (
                f"{user_prompt}\n\n{last_error}. Chunk the task into a "
                f"smaller plan."
            )
            continue

        return plan

    return {
        "status": "error",
        "kind": "planner_failed",
        "detail": (
            f"Planner failed to produce a valid plan after {max_retries + 1} "
            f"attempts. Last error: {last_error}"
        ),
    }


**Quick check — emit a plan for a simple task:**

```python
task = "Research recent developments in the Model Context Protocol (MCP) and summarize the top 3 key points."
plan = planner_agent(task)
if isinstance(plan, Plan):
    print(f"Plan with {len(plan.steps)} steps:")
    for s in plan.steps:
        deps = f" (depends on: {s.depends_on})" if s.depends_on else ""
        pg = f" [parallel_group: {s.parallel_group}]" if s.parallel_group else ""
        print(f"  {s.id}: {s.tool}({s.args}){deps}{pg}")
else:
    print(f"Planner failed: {plan}")
```

A reasonable plan for the MCP task looks like:

```
Plan with 4 steps:
  step_1: web_search({'query': 'Model Context Protocol MCP recent developments', 'recency': 'month'})
  step_2: fetch_page({'url': '<first URL from step_1>'}) (depends on: ['step_1']) [parallel_group: fetches]
  step_3: fetch_page({'url': '<second URL from step_1>'}) (depends on: ['step_1']) [parallel_group: fetches]
  step_4: fetch_page({'url': '<third URL from step_1>'}) (depends on: ['step_1']) [parallel_group: fetches]
```

Notice the URLs are *placeholders* — the planner can't know them in
advance. The executor will resolve them from `step_1`'s actual results.
This is **role-based dependency** in action: the plan says "fetch the
URLs from step_1's results" without committing to specific URLs the
planner had to guess.

(In practice the planner often emits literal placeholder strings like
`"<first URL from step_1>"`. The executor's anti-improvement prompt
handles this by reading the dependency outputs and substituting in the
real values — see Step 4.)

## Step 4: The executor agent

Runs one step at a time. Receives `(step, dependency_outputs)`. The
system prompt is tight and anti-improvement: "run the specified tool
with the specified arguments. If you can't, return `cannot_execute`."

The executor *can* substitute placeholders in `args` with real values
from `dependency_outputs` — that's not "improvement," that's executing
the plan as designed. But it cannot change the tool, add steps, or
override the planner's intent.

In [ ]:
EXECUTOR_SYSTEM_PROMPT = """You are an executor worker. You receive a single
step from a plan, plus the outputs of any steps it depends on. Your job
is to RUN the step's specified tool with its specified arguments.

You may:
- Substitute placeholder values in the step's args with concrete values from
  dependency_outputs (e.g., if step.args.url is "<first URL from step_1>",
  pick the actual first URL from step_1's results).

You MUST NOT:
- Use a different tool than the one specified.
- Add or skip steps.
- "Improve" the args beyond filling in placeholders.

If you genuinely cannot execute the step (the specified tool is missing
necessary inputs, the dependency outputs are unusable, the placeholder
can't be resolved), respond with the JSON:

  {"status": "cannot_execute", "reason": "<specific reason>"}

Otherwise, invoke the tool ONCE and return its result as your final response
with no further tool calls.
"""


def _make_executor_tool_schemas() -> list[dict]:
    """Build the tool schemas the executor sees. Same shape as Lab 10."""
    return [
        {
            "type": "function",
            "function": {
                "name": "web_search",
                "description": EXECUTOR_TOOL_REGISTRY["web_search"]["description"],
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string"},
                        "recency": {"type": "string",
                                    "enum": ["any", "day", "week", "month", "year"]},
                        "max_results": {"type": "integer"},
                    },
                    "required": ["query"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "fetch_page",
                "description": EXECUTOR_TOOL_REGISTRY["fetch_page"]["description"],
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {"type": "string"},
                        "max_chars": {"type": "integer"},
                    },
                    "required": ["url"],
                },
            },
        },
    ]


def _execute_tool(name: str, args: dict) -> dict:
    """Lab 10/11's executor dispatcher; unchanged."""
    if name == "web_search":
        return web_search(
            args.get("query", ""),
            args.get("recency", "any"),
            args.get("max_results", 8),
        )
    if name == "fetch_page":
        return fetch_page(args.get("url", ""), args.get("max_chars", 8000))
    return {"status": "error", "kind": "unknown_tool", "detail": name}


def executor_agent(step: PlanStep, dependency_outputs: dict) -> dict:
    """Run one plan step. Return its result envelope."""
    # The executor sees the step + the resolved dependencies, nothing else.
    deps_summary = json.dumps(dependency_outputs, indent=2)[:4000]
    user_prompt = (
        f"STEP TO EXECUTE:\n"
        f"  id: {step.id}\n"
        f"  description: {step.description}\n"
        f"  tool: {step.tool}\n"
        f"  args: {json.dumps(step.args)}\n\n"
        f"DEPENDENCY OUTPUTS (use these to resolve any placeholders in args):\n"
        f"{deps_summary}\n\n"
        f"Call the specified tool ONCE with the resolved arguments."
    )

    messages: list[dict] = [
        {"role": "system", "content": EXECUTOR_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    tools = _make_executor_tool_schemas()

    for _ in range(EXECUTOR_MAX_STEPS):
        msg = chat_with_tools(messages, tools=tools)
        # Add the assistant turn to the conversation
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        # Check for cannot_execute signal
        if not msg.tool_calls:
            raw = _strip_code_fences(msg.content or "")
            try:
                obj = json.loads(raw)
                if isinstance(obj, dict) and obj.get("status") == "cannot_execute":
                    return obj
            except json.JSONDecodeError:
                pass
            # No tool call and no cannot_execute envelope — treat as failure
            return {
                "status": "error", "kind": "no_tool_call",
                "detail": "executor returned no tool call and no structured signal",
            }

        # Take only the first tool call; we enforce one-tool-per-step
        tc = msg.tool_calls[0]

        # Validate the tool matches the step's declared tool
        if tc.name != step.tool:
            return {
                "status": "error", "kind": "wrong_tool",
                "detail": (f"executor used {tc.name} but step.tool was {step.tool}; "
                           f"this violates the anti-improvement rule"),
            }

        result = _execute_tool(tc.name, tc.arguments)
        return {"status": "ok", "tool": tc.name, "args": tc.arguments, "result": result}

    return {
        "status": "error", "kind": "executor_step_cap",
        "detail": f"executor exceeded EXECUTOR_MAX_STEPS={EXECUTOR_MAX_STEPS}",
    }


## Step 5: The dependency-resolving dispatcher

This is the new core component — and it's pure Python, no LLM. Once
the plan exists, dispatching is mechanical: compute ready steps, submit
them to the pool, collect results, repeat.

Why no LLM here? Because dispatching doesn't need judgment. The plan
already encoded the dependencies; an LLM in this layer would only add
cost and a new failure mode (the LLM mis-resolving dependencies).
Keeping the dispatcher mechanical is the same discipline as keeping the
critic stateless in Lab 11.

In [ ]:
def _ready_steps(
    plan: Plan,
    completed: dict[str, dict],
    failed: set[str],
    in_flight: set[str],
) -> list[PlanStep]:
    """Steps whose dependencies are all completed and that aren't running."""
    ready = []
    for s in plan.steps:
        if s.id in completed or s.id in in_flight or s.id in failed:
            continue
        if all(dep in completed for dep in s.depends_on):
            ready.append(s)
    return ready


def _group_parallel(steps: list[PlanStep]) -> list[list[PlanStep]]:
    """Group ready steps by parallel_group; ungrouped steps run alone."""
    groups: dict[str, list[PlanStep]] = {}
    singletons: list[list[PlanStep]] = []
    for s in steps:
        if s.parallel_group is None:
            singletons.append([s])
        else:
            groups.setdefault(s.parallel_group, []).append(s)
    return list(groups.values()) + singletons


def dispatch_plan(plan: Plan, verbose: bool = True) -> dict:
    """Execute all steps in plan, respecting dependencies + parallel groups.

    Returns:
      {
        "status": "ok" | "partial",
        "results": {step_id: executor_result, ...},
        "failed": [step_id, ...],
        "timing": {"wall_clock_s": float},
      }
    """
    completed: dict[str, dict] = {}
    failed_steps: set[str] = set()
    failed_records: list[dict] = []
    in_flight: set[str] = set()
    lock = threading.Lock()
    start = time.time()

    seen_action_hashes: set[str] = set()

    def _run_step(step: PlanStep) -> tuple[str, dict]:
        # Resolve dependency outputs (read-only access; lock for safety)
        with lock:
            dep_outputs = {dep: completed[dep] for dep in step.depends_on}

        # Action-hash dedup at the executor pool level
        action_sig = _action_hash(step.tool, step.args)
        if action_sig in seen_action_hashes:
            return step.id, {
                "status": "error", "kind": "duplicate_action",
                "detail": f"step {step.id} matches a previously-run action signature",
            }
        seen_action_hashes.add(action_sig)

        result = executor_agent(step, dep_outputs)
        return step.id, result

    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_EXECUTORS) as pool:
        while True:
            ready = _ready_steps(plan, completed, failed_steps, in_flight)
            if not ready and not in_flight:
                break

            # Submit ready steps (subject to pool capacity)
            futures = []
            for step in ready:
                if len(in_flight) >= MAX_PARALLEL_EXECUTORS:
                    break
                in_flight.add(step.id)
                if verbose:
                    pg = f"[{step.parallel_group}]" if step.parallel_group else ""
                    print(f"  → dispatch {step.id}: {step.tool}({step.args}) {pg}")
                futures.append(pool.submit(_run_step, step))

            if not futures and in_flight:
                # Wait for at least one to finish before checking ready again.
                # We can't submit more until in_flight has capacity.
                time.sleep(0.05)
                continue

            for fut in as_completed(futures):
                step_id, result = fut.result()
                with lock:
                    in_flight.discard(step_id)
                    if result.get("status") == "ok":
                        completed[step_id] = result
                        if verbose:
                            print(f"  ← {step_id}: ok")
                    else:
                        failed_steps.add(step_id)
                        failed_records.append({"step_id": step_id, **result})
                        if verbose:
                            print(f"  ✗ {step_id}: {result.get('status')} "
                                  f"({result.get('kind', '?')})")

    elapsed = time.time() - start
    return {
        "status": "ok" if not failed_steps else "partial",
        "results": completed,
        "failed": failed_records,
        "timing": {"wall_clock_s": round(elapsed, 2)},
    }


## Step 6: The replanning hook

When `dispatch_plan` returns with failures, the supervisor invokes the
planner *again* with the failure context. `MAX_REPLANS = 2`. If the
replan cap fires, surface the partial results honestly.

A subtle but important guard: if the replanner produces an *identical*
plan to the one that just failed, we treat that as escalation —
producing the same plan again won't help. Same Lab 03/10/11 dedup
discipline applied to plans.

In [ ]:
def _plan_signature(plan: Plan) -> str:
    """Stable signature of a plan, for dedup."""
    # Only the structural content; not the descriptions (model variations).
    structural = [
        {"id": s.id, "tool": s.tool, "args": s.args,
         "depends_on": sorted(s.depends_on),
         "parallel_group": s.parallel_group}
        for s in plan.steps
    ]
    return hashlib.sha256(
        json.dumps(structural, sort_keys=True).encode()
    ).hexdigest()[:16]


def plan_and_execute(task: str, verbose: bool = True) -> dict:
    """Top-level: plan → execute → maybe replan → return."""
    seen_plan_sigs: set[str] = set()
    replan_count = 0
    last_partial: dict | None = None

    failure_context: dict | None = None

    while True:
        if verbose:
            if failure_context:
                print(f"\n── Replanning (attempt {replan_count + 1}/{MAX_REPLANS}) ──")
            else:
                print("\n── Planning ──")

        plan_or_err = planner_agent(task, failure_context=failure_context)
        if not isinstance(plan_or_err, Plan):
            return {
                "status": "error",
                "stage": "planner",
                "detail": plan_or_err,
                "partial": last_partial,
            }
        plan = plan_or_err
        sig = _plan_signature(plan)
        if sig in seen_plan_sigs:
            return {
                "status": "error",
                "stage": "replanner_duplicate",
                "detail": "Replanner produced an identical plan; escalating.",
                "plan": [s.model_dump() for s in plan.steps],
                "partial": last_partial,
            }
        seen_plan_sigs.add(sig)

        if verbose:
            print(f"  Plan has {len(plan.steps)} steps")

        if verbose:
            print("\n── Executing ──")
        exec_result = dispatch_plan(plan, verbose=verbose)
        last_partial = exec_result

        if exec_result["status"] == "ok":
            return {
                "status": "ok",
                "plan": [s.model_dump() for s in plan.steps],
                "execution": exec_result,
                "replans": replan_count,
            }

        # Partial; consider replan
        if replan_count >= MAX_REPLANS:
            return {
                "status": "partial_after_cap",
                "plan": [s.model_dump() for s in plan.steps],
                "execution": exec_result,
                "replans": replan_count,
                "detail": (
                    f"Hit MAX_REPLANS={MAX_REPLANS}. Surfacing the latest "
                    f"partial result honestly."
                ),
            }

        # Set up failure context for the replanner
        first_failure = exec_result["failed"][0]
        failure_context = {
            "step_id": first_failure["step_id"],
            "error": first_failure.get("detail") or first_failure.get("kind"),
            "completed_steps": list(exec_result["results"].keys()),
        }
        replan_count += 1


## Step 7: The synthesizer (final-answer composer)

After execution, the synthesizer takes the original task + the step
results and produces the final answer. It's a writer-style agent — no
tools, one LLM call, prose output with citation preservation when
fetched pages produced URLs.

The synthesizer's input is structured: `{task, step_results}`. Same
handoff-hygiene discipline as Lab 10/11.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """You are a synthesizer agent. You receive the
original user task plus a dict of executed step results from a plan.

Your job: produce a clear, accurate answer to the user task using only
information from the step results.

Rules:
1. Cite sources. For every claim drawn from a fetched page, reference its
   URL inline as [1], [2], etc., and list the citations at the end:
       [1] Title — URL

2. Do not invent claims not supported by the step results.

3. If a step failed or returned no useful information, say so. Do not
   paper over gaps.

4. Keep the final answer focused on what the user asked for, not on
   how the plan worked.
"""


def synthesizer_agent(task: str, execution: dict) -> dict:
    """Compose the final answer."""
    results = execution.get("results", {})
    failed = execution.get("failed", [])

    # Render step results compactly for the synthesizer's prompt
    step_summary_lines = []
    citations_so_far: list[dict] = []
    for step_id, step_result in results.items():
        tool = step_result.get("tool")
        result = step_result.get("result", {})
        # Truncate large results
        result_str = json.dumps(result)[:3000]
        step_summary_lines.append(f"--- {step_id} ({tool}) ---\n{result_str}")

        # Pull citations from successful fetch_page results
        if tool == "fetch_page" and result.get("status") in ("ok", "too_long"):
            citations_so_far.append({
                "url": result["url"],
                "title": result.get("title", ""),
            })

    step_summary = "\n\n".join(step_summary_lines) or "(no completed steps)"
    failure_summary = ""
    if failed:
        failure_lines = [
            f"- {f['step_id']}: {f.get('kind')} ({f.get('detail', '')[:200]})"
            for f in failed
        ]
        failure_summary = "\n\nFAILED STEPS:\n" + "\n".join(failure_lines)

    user_prompt = (
        f"USER TASK:\n{task}\n\n"
        f"COMPLETED STEP RESULTS:\n{step_summary}"
        f"{failure_summary}\n\n"
        f"Compose the final answer. Cite fetched URLs inline."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": SYNTHESIZER_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return {
        "status": "ok",
        "answer": msg.content or "",
        "citations": citations_so_far,
    }


## Step 8: Run the full pipeline end-to-end

A real task: research recent MCP developments, summarize three key
points. The plan should have search + parallel fetches + (possibly) a
serial structuring step. Expected wall-clock: ~10-20 seconds with
parallelism, ~25-40 seconds without.

In [ ]:
def run_plan_and_execute(task: str, verbose: bool = True) -> dict:
    """Top-level entry point."""
    result = plan_and_execute(task, verbose=verbose)

    if result["status"] == "error":
        return {
            "status": "error",
            "detail": result.get("detail"),
            "partial": result.get("partial"),
            "answer": None,
        }

    if verbose:
        print("\n── Synthesizing ──")
    synth = synthesizer_agent(task, result["execution"])

    return {
        "status": result["status"],
        "answer": synth["answer"],
        "citations": synth["citations"],
        "plan": result["plan"],
        "replans": result.get("replans", 0),
        "wall_clock_s": result["execution"]["timing"]["wall_clock_s"],
        "failed_steps": result["execution"].get("failed", []),
    }


task = (
    "Research recent developments in the Model Context Protocol (MCP). "
    "Summarize the top 3 key points with citations."
)
final = run_plan_and_execute(task)

print("\n" + "=" * 70)
print("FINAL ANSWER:")
print("=" * 70)
print(final["answer"])
print()
print(f"Status: {final['status']}, replans: {final['replans']}, "
      f"wall_clock: {final['wall_clock_s']}s")
print(f"Plan had {len(final['plan'])} steps; "
      f"{len(final['failed_steps'])} failed")


**Sample output (LLM responses will vary; trajectory should be stable):**

```
── Planning ──
  Plan has 4 steps

── Executing ──
  → dispatch step_1: web_search({'query': 'Model Context Protocol MCP', 'recency': 'month'})
  ← step_1: ok
  → dispatch step_2: fetch_page({'url': '<first URL from step_1>'}) [fetches]
  → dispatch step_3: fetch_page({'url': '<second URL from step_1>'}) [fetches]
  → dispatch step_4: fetch_page({'url': '<third URL from step_1>'}) [fetches]
  ← step_2: ok
  ← step_3: ok
  ← step_4: ok

── Synthesizing ──

======================================================================
FINAL ANSWER:
======================================================================
The Model Context Protocol (MCP) [1] is an open standard introduced by
Anthropic in late 2024. Three recent developments stand out:

1. Expanded server ecosystem...
2. Production deployments...
3. Tool authentication patterns...

[1] Introducing the Model Context Protocol — https://www.anthropic.com/news/...
[2] MCP Specification — https://modelcontextprotocol.io/...
[3] MCP server gallery — https://...

Status: ok, replans: 0, wall_clock: 6.4s
Plan had 4 steps; 0 failed
```

Three parallel fetches in ~6 seconds. Sequentially they'd take 12-15 seconds.
The parallelism gain is real.

## Step 9: Failure-mode walkthrough

The four plan-and-execute-specific failure modes, with the mitigation
Lab 12 ships against each.

### Failure mode 1: Plan brittleness

**Symptom**: planner emits steps that depend on specific structures it
imagined (e.g., "fetch the third result") that don't hold at execution
time.

**Mitigation**: Step descriptions use role-based references, and the
executor sees the actual dependency outputs. The planner can say
"fetch URLs from step_1's results" without committing to specific
URLs.

**Diagnostic**: Look at the executor agent — it gets `dep_outputs` and
the placeholder-style args. It substitutes real values at runtime, so
the plan survives even when the planner couldn't predict result
structure.

In [ ]:
# Demonstrate: planner can emit placeholder args; executor resolves them.
# Here's a fabricated step + dep_outputs to show what the executor sees:

example_step = PlanStep(
    id="step_2", description="Fetch the first URL from step_1's results.",
    tool="fetch_page", args={"url": "<first URL from step_1>"},
    depends_on=["step_1"], parallel_group="fetches",
)
example_deps = {
    "step_1": {
        "tool": "web_search",
        "result": {
            "status": "ok",
            "results": [
                {"title": "MCP overview", "url": "https://example.com/mcp"},
                {"title": "Tool spec", "url": "https://example.com/spec"},
            ],
        },
    },
}
# The executor sees this and substitutes the placeholder with the actual URL.
# (We don't actually run a fetch here to avoid live web traffic in this cell.)
print(f"Step args: {example_step.args}")
print(f"Dep outputs (first URL): {example_deps['step_1']['result']['results'][0]['url']}")
print("\nThe executor would substitute the placeholder with the real URL")
print("at execution time — that's the mitigation for plan brittleness.")


### Failure mode 2: Execution drift

**Symptom**: executor decides a different tool or different arguments
would be better and uses them. Downstream steps break.

**Mitigation**: The executor's system prompt explicitly forbids tool
substitution. The `executor_agent` function additionally validates
that `tc.name == step.tool` and returns a `wrong_tool` error envelope
if the executor tries to deviate.

In [ ]:
# The executor enforces tool-name matching after the LLM emits a tool call.
# Show the validation logic by direct check:

example_step = PlanStep(id="step_1", description="x", tool="web_search",
                       args={"query": "test"})
# If the LLM (hypothetically) tried to use fetch_page when the step said
# web_search, the executor would catch it:

simulated_tool_call_name = "fetch_page"  # Wrong!
expected_tool = example_step.tool
if simulated_tool_call_name != expected_tool:
    print(f"DETECTED: executor tried '{simulated_tool_call_name}' but step "
          f"specified '{expected_tool}'")
    print("→ returned error envelope: status=error, kind=wrong_tool")


### Failure mode 3: Replanning thrash

**Symptom**: each executor failure triggers a full replan; new plan
fails the same way; never converges.

**Mitigation**: `MAX_REPLANS = 2` + identical-plan dedup. The
`_plan_signature` helper computes a stable structural hash of the
plan; identical signatures across replans escalates to `partial_after_cap`.

In [ ]:
# Demonstrate the plan signature dedup:
plan_a = Plan(steps=[
    PlanStep(id="step_1", description="x", tool="web_search",
             args={"query": "test"}),
])
plan_b = Plan(steps=[
    # Same structural content, different description
    PlanStep(id="step_1", description="y (different wording)",
             tool="web_search", args={"query": "test"}),
])
plan_c = Plan(steps=[
    PlanStep(id="step_1", description="x", tool="web_search",
             args={"query": "different query"}),  # Genuinely different
])

sig_a = _plan_signature(plan_a)
sig_b = _plan_signature(plan_b)
sig_c = _plan_signature(plan_c)
print(f"Plan A: {sig_a}")
print(f"Plan B (different description, same structure): {sig_b}  "
      f"{'same' if sig_a == sig_b else 'diff'}")
print(f"Plan C (different args):                        {sig_c}  "
      f"{'same' if sig_a == sig_c else 'diff'}")
print("\nSignature ignores prose differences, catches structural sameness.")
print("If the replanner returns plan_b after plan_a failed, we escalate.")


### Failure mode 4: Plan-execution gap

**Symptom**: planner emits steps using tools the executor doesn't have.
Plan looks reasonable; can't run.

**Mitigation**: The executor's tool registry is passed into the
planner's system prompt; `Plan.validate_graph()` rejects plans
referencing unknown tools; the planner-retry loop in Step 3 surfaces
the error back to the planner with the specific failure.

In [ ]:
# Show the validation catching a plan-execution gap:
bad_plan = Plan(steps=[
    PlanStep(id="step_1", description="Query the SQL database for users.",
             tool="sql_query", args={"query": "SELECT * FROM users"}),
])

executor_tools = set(EXECUTOR_TOOL_REGISTRY.keys())
errors = bad_plan.validate_graph(executor_tools)
print(f"Validation errors: {errors}")
print(f"\nExecutor only has: {sorted(executor_tools)}")
print("The planner's retry loop would surface this error back to it,")
print("constraining the next attempt to available tools.")


## Step 10 (stretch): Plan-and-execute vs. ReAct

Both can solve research tasks. Different shape, different costs,
different bug profiles. The honest comparison:

**Plan-and-execute** (this lab): plans upfront → executes → maybe
replans. Auditable plan as artifact. Can parallelize. Brittle when the
task can't be decomposed upfront.

**ReAct** (Lab 03's pattern): think → act → observe → think → act → ...
No plan; trajectory emerges step-by-step. Robust to surprises; adapts
each step based on what the previous step returned. Sequential by
construction. No artifact to audit before execution.

Empirically, for tasks like "research X and summarize" with a clear
decomposition, plan-and-execute wins on:

- Wall-clock (parallelism on fetches)
- Auditability (plan exists as a structured artifact)
- Cost variance (typically 8-12 calls; ReAct can range 4-20)

ReAct wins on:

- Adaptive tasks ("find me an answer to X" where the answer isn't a
  known decomposition)
- Lower minimum cost (4-6 calls for simple tasks vs ~8 for
  plan-and-execute's planner + executors + synthesizer)
- Better recovery from surprises (each step's observation informs the
  next; plan-and-execute has to replan)

A useful framing: **plan-and-execute optimizes the median run** (more
auditable, parallelizable, predictable); **ReAct optimizes the worst-
case run** (more adaptive, fewer brittle assumptions).

Try the same task with both patterns on your own — Lab 03 has the
ReAct agent; this notebook has plan-and-execute. Compare wall-clock,
total LLM calls, and final-answer quality. The "right" pattern depends
on which dimension you care about most for your production constraint.

## What you just built

A plan-and-execute system in ~400 lines of Python total:

- `PlanStep` and `Plan` Pydantic schemas with dependency-graph
  validation and parallel-group integrity checks.
- A planner agent that emits JSON-validated plans, with retry loops
  for malformed output.
- An executor agent that runs one step at a time with anti-improvement
  framing.
- A dependency-resolving dispatcher using `ThreadPoolExecutor` with
  bounded concurrency (`MAX_PARALLEL_EXECUTORS = 3`).
- A bounded replanning hook (`MAX_REPLANS = 2`) with identical-plan
  dedup to prevent thrash.
- A synthesizer that composes the final answer with citation
  preservation.
- Four explicit failure-mode mitigations: role-based descriptions
  (plan brittleness), tool-name validation (execution drift), plan
  signature dedup + cap (replanning thrash), executor-registry-in-
  planner-prompt (plan-execution gap).

The reuse is real: the chat client, web tools, action-hash dedup,
StrictModel pattern, and structured-error envelopes all carry over
unchanged from Lab 10/11. **Plan-and-execute turns out to be a few
new schemas, a new agent role, and a Python dispatcher — not a new
framework.** That's the bet of Path 03.

## Production readiness — out of scope here

For a real deployment you'd also want: distributed execution
(`ProcessPoolExecutor` or a queue + workers); persistent plan state
(LangGraph's checkpointer pattern); per-step cost budgets and global
budget enforcement; observability hooks (OTel) at the dispatcher;
plan-quality eval (does the planner produce plans that complete
without replan? what's the replan rate?); critic on the plan
(generator-critic from Lab 11 applied at the planning stage).

## Next

- Take the [plan-and-execute quiz](../../quizzes/multi-agent/plan-and-execute.md).
- Path 03 continues with Module 4 (multi-agent RAG) in a future batch.
- If you've also done Path 02, the multi-agent RAG batch will compose
  Lab 06-08's retrieval pipeline with Labs 10-12's coordination
  patterns.